In [ ]:
import pandas as pd
import datetime as dt
import numpy as np 
import glob
import matplotlib.pyplot as plt 
import matplotlib.patches as patches

from matplotlib.ticker import AutoMinorLocator


from JunoPy.spice import InitSpiceKernels, get_juno_position
from JunoPy.mag import MagData, mva

# Get Juno trajectory in JUNO_MAG_VIP4 coordinate
InitSpiceKernels()
# start = dt.datetime.strptime('2017-219T00:00:00', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2017-271T23:59:59', '%Y-%jT%H:%M:%S')
start = dt.datetime.strptime('2017-230T00:00:00', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2017-250T23:59:59', '%Y-%jT%H:%M:%S')
time_array = pd.date_range(
        start=start, end=end, periods=10000).to_pydatetime().tolist()
position1, _ = get_juno_position(time_array, coordsystem='JUNO_MAG_VIP4')

start = dt.datetime.strptime('2017-325T00:00:00', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2018-012T23:59:59', '%Y-%jT%H:%M:%S')
time_array = pd.date_range(
        start=start, end=end, periods=10000).to_pydatetime().tolist()
position2, _ = get_juno_position(time_array, coordsystem='JUNO_MAG_VIP4')

# Get Juno position for the selected events
def read_csv_file():
    starttimes = []
    endtimes = []
    durations = []
    PNevents = []
    NPevents = []
    with open('mva_3events.csv', 'r') as f:
        f.readline()
        
        for i, line in enumerate(f):    
            starttimes.append(dt.datetime.strptime(line.split(',')[0], '%Y-%jT%H:%M:%S'))
            endtimes.append(dt.datetime.strptime(line.split(',')[1], '%Y-%jT%H:%M:%S'))
            durations.append(float(line.split(',')[2]))
            if 'PN' in line.split(',')[3]:
                PNevents.append(i)
            else:
                NPevents.append(i)
            
    return starttimes, endtimes, durations, PNevents, NPevents
            
starttimes, endtimes, durations, PNevents, NPevents = read_csv_file()
InitSpiceKernels()
position_mva, _ = get_juno_position(starttimes, coordsystem='JUNO_MAG_VIP4')

In [ ]:
# Plot the trajectory
fig = plt.figure(figsize=(6,8), dpi=600)
ax = plt.subplot(111)
ax.plot(np.sqrt(position1[:, 0]**2 + position1[:, 1]**2), 
        position1[:, 2], 
        zorder=1, lw=1.0, label='Perijove 8', color='b') # alpha=0.5, color=[1.0, 0.3, 0.3]
# ax.plot(np.sqrt(position2[:, 0]**2 + position2[:, 1]**2), 
#         position2[:, 2], 
#         color=[0.3, 0.3, 1.0], zorder=1, alpha=0.5, lw=1.0, label='Perijove 10')

# sc = ax.scatter(np.sqrt(position_mva[:, 0]**2 + position_mva[:, 1]**2),
#                 position_mva[:, 2], c='0.0',
#                 zorder=2, marker='o', cmap='jet')

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.grid('on', which='minor', lw=0.5, ls='dotted')
ax.grid(color='k', ls='dashed', lw=0.25)

ax.set_xlabel('$\sqrt{x^2 + y^2}$ [R$_J$]')
ax.set_ylabel('z [R$_J$]')
ax.legend(loc='upper left')

ax.set_xlim([0, 100])
ax.set_ylim([-40, 20])
ax.set_aspect('equal', share=True)

# ax.text(75, -10, '#1', color='k', zorder=3, weight='bold')
# ax.text(90, -3, '#2', color='k', zorder=3, weight='bold')

style="Simple,tail_width=1.0,head_width=4,head_length=8"
kw = dict(arrowstyle=style, color="k")
a1 = patches.FancyArrowPatch((60, 13), (50, 13),connectionstyle="arc3,rad=.05", **kw)
a2 = patches.FancyArrowPatch((20, -20), (30, -25),connectionstyle="arc3,rad=.05", **kw)
for a in [a1, a2]:
    ax.add_patch(a)


ax.text(61, 12, 'Juno Trajectory', color='k', zorder=3)

ax.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
fig.savefig('out_images/TrajectoryLocationofEvent_magvip4_oneperijove.png')